# Advanced Problems with Solutions: Python Class Body Scope

This notebook is a deep practice set on **class body scope, name resolution, methods, closures, comprehensions, nested classes, decorators, default arguments, and `__class__` cells**.

It is designed to be run **top-to-bottom**. Cells that demonstrate errors catch them explicitly so the notebook keeps running.

## Core mental model

When Python executes a `class` statement:

1. The **class body executes immediately** in its own namespace.
2. Expressions directly in the class body can usually see names already stored in that class namespace.
3. A function defined in the class body is **not lexically enclosed by the class namespace**.
4. Therefore, an unqualified name used inside a method follows normal function lookup rules rather than "look in my class first".
5. Use `self.attr`, `cls.attr`, or an explicit class name when you mean a class/instance attribute.
6. Comprehensions have their own function-like scope, which creates subtle class-body behavior.
7. A class created inside a function can contain methods that close over the function's local variables.

The goal of the exercises is not merely to memorize rules, but to be able to **predict code before running it**.


## Best-practice checklist

Use these rules in production code:

- Prefer `self.attr` for instance-oriented behavior.
- Prefer `cls.attr` inside class methods.
- Avoid relying on same-named globals from methods.
- Do not depend on class-body comprehension quirks for important logic.
- Use explicit factory parameters when a generated class intentionally captures configuration.
- Keep class constants visually distinct, commonly with `UPPER_CASE`.
- Write a regression test whenever name resolution is subtle enough to surprise a reviewer.
- Prefer code whose scope behavior is obvious without disassembly.


In [1]:
import inspect
import dis
from pprint import pprint

print("Notebook helpers imported.")


Notebook helpers imported.


# Warm-up: direct class-body lookup vs method lookup


In [2]:
GLOBAL_LABEL = "module"

class WarmUp:
    LABEL = "class"

    direct = LABEL

    def method(self):
        # LABEL is NOT looked up in the class namespace.
        return GLOBAL_LABEL

print(WarmUp.direct)
print(WarmUp().method())

assert WarmUp.direct == "class"
assert WarmUp().method() == "module"


class
module


# Problem 1 — Predict four different lookups

Without running the next cell first, predict the four values produced by:

- `Config.snapshot`
- `Config().via_instance()`
- `Config.via_class()`
- `Config.via_global()`

Then run the cell and explain why all four are not the same.


In [3]:
VERSION = "global-1.0"

class Config:
    VERSION = "class-2.0"
    snapshot = VERSION

    def via_instance(self):
        return self.VERSION

    @classmethod
    def via_class(cls):
        return cls.VERSION

    @staticmethod
    def via_global():
        return VERSION

results = {
    "snapshot": Config.snapshot,
    "instance": Config().via_instance(),
    "class": Config.via_class(),
    "global": Config.via_global(),
}
pprint(results)


{'class': 'class-2.0',
 'global': 'global-1.0',
 'instance': 'class-2.0',
 'snapshot': 'class-2.0'}


### Solution 1

Expected:

```python
{
    "snapshot": "class-2.0",
    "instance": "class-2.0",
    "class": "class-2.0",
    "global": "global-1.0",
}
```

`Config.snapshot = VERSION` runs directly in the class body, so `VERSION` resolves in the class namespace.  
The instance and class methods explicitly use `self.VERSION` and `cls.VERSION`.  
The static method uses the unqualified name `VERSION`, so normal function/global lookup finds the module-level variable.


In [4]:
assert results == {
    "snapshot": "class-2.0",
    "instance": "class-2.0",
    "class": "class-2.0",
    "global": "global-1.0",
}
print("Problem 1 passed.")


Problem 1 passed.


# Problem 2 — Diagnose a dangerous bug that does not raise an exception

The method below is intended to return the class's retry count.

Why can this bug survive tests if a module-level variable happens to have the same name?
Fix it using the most appropriate lookup style.


In [5]:
RETRIES = 99

class Client:
    RETRIES = 3

    @classmethod
    def retry_count_buggy(cls):
        return RETRIES

print("Buggy value:", Client.retry_count_buggy())


Buggy value: 99


### Solution 2

The code does not raise `NameError` because `RETRIES` exists globally. The method silently returns the wrong value.

Inside a class method, use `cls.RETRIES`.


In [6]:
class Client:
    RETRIES = 3

    @classmethod
    def retry_count(cls):
        return cls.RETRIES

assert Client.retry_count() == 3
print("Fixed value:", Client.retry_count())


Fixed value: 3


# Problem 3 — Inheritance and why `cls` is better than a hard-coded class name

Implement `describe()` so subclasses automatically report their own overridden `FORMAT`.


In [7]:
class Serializer:
    FORMAT = "generic"

    @classmethod
    def describe(cls):
        # TODO solved below:
        return f"{cls.__name__} uses {cls.FORMAT}"

class JsonSerializer(Serializer):
    FORMAT = "json"

class XmlSerializer(Serializer):
    FORMAT = "xml"

print(Serializer.describe())
print(JsonSerializer.describe())
print(XmlSerializer.describe())

assert JsonSerializer.describe() == "JsonSerializer uses json"
assert XmlSerializer.describe() == "XmlSerializer uses xml"


Serializer uses generic
JsonSerializer uses json
XmlSerializer uses xml


### Solution 3

`cls.FORMAT` is polymorphic: when a subclass calls the inherited class method, `cls` is that subclass.

A hard-coded expression such as `Serializer.FORMAT` would defeat overriding and always read the base class.


# Problem 4 — Method closes over the factory function, not the class namespace

Predict the result of `Generated.report()`.

Pay attention to the three different `LEVEL` variables.


In [8]:
LEVEL = "module"

def make_class():
    LEVEL = "factory"

    class Generated:
        LEVEL = "class"

        @classmethod
        def report(cls):
            return LEVEL

    return Generated

Generated = make_class()

print("Generated.LEVEL:", Generated.LEVEL)
print("Generated.report():", Generated.report())


Generated.LEVEL: class
Generated.report(): factory


### Solution 4

`Generated.LEVEL` is `"class"`, but `Generated.report()` is `"factory"`.

The method is defined while `make_class()` is running, so the method can close over the function-local `LEVEL`.  
The class namespace itself is not the method's lexical enclosing scope.


In [9]:
assert Generated.LEVEL == "class"
assert Generated.report() == "factory"

closure_info = inspect.getclosurevars(Generated.report)
print(closure_info)
assert closure_info.nonlocals["LEVEL"] == "factory"


ClosureVars(nonlocals={'LEVEL': 'factory'}, globals={}, builtins={}, unbound=set())


# Problem 5 — Build a configurable class factory intentionally

Write a factory `make_validator(min_value, max_value)` that returns a class with:

- class attributes `MIN_VALUE` and `MAX_VALUE`
- an instance method `valid(value)` that intentionally uses captured factory arguments
- a class method `bounds()` that reads the class attributes through `cls`

Then compare the two mechanisms.


In [10]:
def make_validator(min_value, max_value):
    class Validator:
        MIN_VALUE = min_value
        MAX_VALUE = max_value

        def valid(self, value):
            # Intentional closure over factory variables.
            return min_value <= value <= max_value

        @classmethod
        def bounds(cls):
            return cls.MIN_VALUE, cls.MAX_VALUE

    return Validator

Percent = make_validator(0, 100)
Temperature = make_validator(-50, 60)

tests = [
    Percent().valid(50),
    not Percent().valid(101),
    Temperature().valid(-20),
    Percent.bounds() == (0, 100),
    Temperature.bounds() == (-50, 60),
]

print(tests)
assert all(tests)


[True, True, True, True, True]


### Solution 5

This is a legitimate use of closure behavior.

However, notice that `valid()` and `bounds()` obtain configuration through different mechanisms:

- `valid()` uses the captured factory values.
- `bounds()` uses class attributes.

If class attributes may later be modified or overridden in subclasses, prefer `self.MIN_VALUE` / `self.MAX_VALUE` so the runtime behavior follows the class hierarchy.


# Problem 6 — Mutation after class creation exposes closure-vs-class differences

Predict the two outputs after changing `Percent.MAX_VALUE` to `200`.


In [11]:
Percent.MAX_VALUE = 200

print("valid(150):", Percent().valid(150))
print("bounds():", Percent.bounds())


valid(150): False
bounds(): (0, 200)


### Solution 6

`Percent().valid(150)` is still `False` because the method captured the original factory value `max_value == 100`.

`Percent.bounds()` returns `(0, 200)` because it reads current class attributes.

This is a strong reason to avoid accidentally mixing closure-backed configuration with mutable class configuration.


In [12]:
assert Percent().valid(150) is False
assert Percent.bounds() == (0, 200)


# Problem 7 — Class-body comprehension trap

Predict `names_direct` and `names_comprehension`.

Why do they resolve `name` differently?


In [13]:
name = "global-name"

class ScopeDemo:
    name = "class-name"

    names_direct = [name] * 3
    names_comprehension = [name for _ in range(3)]

print("direct:", ScopeDemo.names_direct)
print("comprehension:", ScopeDemo.names_comprehension)


direct: ['class-name', 'class-name', 'class-name']
comprehension: ['global-name', 'global-name', 'global-name']


### Solution 7

Expected:

```python
["class-name", "class-name", "class-name"]
["global-name", "global-name", "global-name"]
```

The direct expression executes in the class body.  
The comprehension body executes in its own function-like scope, so the class namespace is not a lexical enclosing scope for `name`.


In [14]:
assert ScopeDemo.names_direct == ["class-name"] * 3
assert ScopeDemo.names_comprehension == ["global-name"] * 3


# Problem 8 — The advanced comprehension nuance: the iterable expression

A comprehension's *body* has its own scope, but the outermost iterable expression is prepared from the surrounding context.

Predict whether this class definition succeeds.


In [15]:
class RangeDemo:
    count = 4

    # `count` is used to construct the iterable.
    values = [i * i for i in range(count)]

print(RangeDemo.values)
assert RangeDemo.values == [0, 1, 4, 9]


[0, 1, 4, 9]


### Solution 8

It succeeds.

`range(count)` can use `count` from the class-body execution context.  
But if the comprehension expression itself tried to use an unqualified class-local name, that name would be resolved from the comprehension's function-like scope instead.

This difference is subtle enough that production code should usually avoid depending on it.


# Problem 9 — Safely demonstrate a failing comprehension without breaking the notebook

The class below has no matching global `factor`.

Catch and inspect the error produced by trying to use class-local `factor` inside the comprehension expression.


In [16]:
globals().pop("factor", None)

try:
    class BrokenComprehension:
        factor = 10
        values = [factor * i for i in range(3)]
except NameError as exc:
    print(type(exc).__name__ + ":", exc)
else:
    raise AssertionError("Expected class creation to fail with NameError")


NameError: name 'factor' is not defined


### Solution 9

The class-local `factor` is not available as an enclosing variable to the comprehension body.

A clear rewrite is to compute after the class is created, use a helper function, or avoid requiring another class-local inside the comprehension.


In [17]:
class ClearComprehension:
    factor = 10

ClearComprehension.values = [
    ClearComprehension.factor * i
    for i in range(3)
]

print(ClearComprehension.values)
assert ClearComprehension.values == [0, 10, 20]


[0, 10, 20]


# Problem 10 — Default arguments are evaluated during class-body execution

Predict whether `DEFAULT_TIMEOUT` is visible in the default value below.
Then explain why the same unqualified name would not be safe inside the method body.


In [18]:
class Request:
    DEFAULT_TIMEOUT = 5

    def send(self, timeout=DEFAULT_TIMEOUT):
        return timeout

r = Request()

print(r.send())
print(r.send(12))

assert r.send() == 5
assert r.send(12) == 12


5
12


### Solution 10

The default expression `DEFAULT_TIMEOUT` is evaluated when the `def` statement is executed, while the class body is being executed. Therefore it can resolve the class-local value at that moment.

The resulting default value is stored on the function object. It is not a live lookup of `Request.DEFAULT_TIMEOUT`.


In [19]:
Request.DEFAULT_TIMEOUT = 30

print("Class attribute now:", Request.DEFAULT_TIMEOUT)
print("Existing method default still:", Request().send())

assert Request.DEFAULT_TIMEOUT == 30
assert Request().send() == 5


Class attribute now: 30
Existing method default still: 5


# Problem 11 — Fix a stale default configuration

The previous example may be undesirable if runtime changes to the class attribute should be respected.

Refactor the method so the default is resolved at call time.


In [20]:
class BetterRequest:
    DEFAULT_TIMEOUT = 5

    def send(self, timeout=None):
        if timeout is None:
            timeout = self.DEFAULT_TIMEOUT
        return timeout

assert BetterRequest().send() == 5

BetterRequest.DEFAULT_TIMEOUT = 30

assert BetterRequest().send() == 30
assert BetterRequest().send(7) == 7

print("Dynamic default:", BetterRequest().send())


Dynamic default: 30


### Solution 11

Using a sentinel such as `None` lets the method perform attribute lookup at call time.

If `None` is itself a valid user value, use a unique sentinel object instead.


In [21]:
_MISSING = object()

class PreciseRequest:
    DEFAULT_TIMEOUT = None

    def send(self, timeout=_MISSING):
        if timeout is _MISSING:
            timeout = self.DEFAULT_TIMEOUT
        return timeout

assert PreciseRequest().send() is None
assert PreciseRequest().send(10) == 10


# Problem 12 — Nested classes do not magically capture the outer class namespace

Predict whether `Inner.copied = token` can see `Outer.token`.

The code catches the expected failure.


In [22]:
globals().pop("token", None)

try:
    class Outer:
        token = "outer-class"

        class Inner:
            copied = token
except NameError as exc:
    print(type(exc).__name__ + ":", exc)
else:
    raise AssertionError("Expected NameError while defining Inner")


NameError: name 'token' is not defined


### Solution 12

The namespace of `Outer` is not a lexical closure for the body of `Inner`.

A nested class is an organizational relationship, not an automatic variable-capture relationship.
Use an explicit reference after the outer class exists, or redesign so the dependency is passed explicitly.


In [23]:
class Outer:
    token = "outer-class"

    class Inner:
        pass

Outer.Inner.copied = Outer.token

assert Outer.Inner.copied == "outer-class"
print(Outer.Inner.copied)


outer-class


# Problem 13 — Decorator lookup happens while the class body executes

Here the decorator name is stored in the class namespace before the method is defined.

Predict whether the decorator can be applied successfully.


In [24]:
def uppercase_result(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

class Greeter:
    decorator = uppercase_result

    @decorator
    def hello(self):
        return "hello from class"

print(Greeter().hello())
assert Greeter().hello() == "HELLO FROM CLASS"


HELLO FROM CLASS


### Solution 13

It works because the decorator expression is evaluated while Python is executing the class body.

This does **not** mean the decorated method's function body is enclosed by the class namespace. These are separate moments:

1. class-body execution evaluates `@decorator`;
2. later, method calls execute the function body under normal function scope rules.


# Problem 14 — The special `__class__` cell

Most class-local names are not lexical parents of methods.  
However, Python has special compiler support for methods that reference `__class__`.

Inspect the closure of the method below.


In [25]:
class Base:
    def defining_class_name(self):
        return __class__.__name__

class Child(Base):
    pass

b = Base()
c = Child()

print(b.defining_class_name())
print(c.defining_class_name())

print("__closure__:", Base.defining_class_name.__closure__)
print("free vars:", Base.defining_class_name.__code__.co_freevars)


Base
Base
__closure__: (<cell at 0x000002D1CE33F700: type object at 0x000002D1BDBD24E0>,)
free vars: ('__class__',)


### Solution 14

Both calls return `"Base"` because `__class__` refers to the class where the method was **defined**, not necessarily the runtime type of `self`.

This compiler-created `__class__` cell is also part of how zero-argument `super()` works.
Use `type(self)` when you want the runtime class of an instance.


In [26]:
assert Base().defining_class_name() == "Base"
assert Child().defining_class_name() == "Base"

class RuntimeAware:
    def runtime_class_name(self):
        return type(self).__name__

class RuntimeChild(RuntimeAware):
    pass

assert RuntimeChild().runtime_class_name() == "RuntimeChild"


# Problem 15 — Connect `__class__` to zero-argument `super()`

Explain why `super()` can work without explicitly passing the class.


In [27]:
class Parent:
    def message(self):
        return "parent"

class ChildWithSuper(Parent):
    def message(self):
        return super().message() + " -> child"

print(ChildWithSuper().message())
print("free vars:", ChildWithSuper.message.__code__.co_freevars)

assert ChildWithSuper().message() == "parent -> child"
assert "__class__" in ChildWithSuper.message.__code__.co_freevars


parent -> child
free vars: ('__class__',)


### Solution 15

The compiler provides a `__class__` closure cell when needed.  
Zero-argument `super()` uses runtime context including that defining-class information and the bound first argument.

This is a special language mechanism; it should not be generalized into the belief that arbitrary class attributes become closure variables.


# Problem 16 — Instance shadowing vs class attributes

This problem combines scope with attribute lookup.

Predict the values before and after assigning `obj.limit = 50`.


In [28]:
class Limits:
    limit = 10

    def via_self(self):
        return self.limit

    @classmethod
    def via_cls(cls):
        return cls.limit

obj = Limits()

before = (obj.limit, obj.via_self(), Limits.via_cls())

obj.limit = 50

after = (obj.limit, obj.via_self(), Limits.via_cls())

print("before:", before)
print("after:", after)


before: (10, 10, 10)
after: (50, 50, 10)


### Solution 16

Expected:

```python
before == (10, 10, 10)
after  == (50, 50, 10)
```

`self.limit` uses normal attribute lookup and therefore sees an instance attribute first.  
`cls.limit` reads from the class hierarchy.

This is attribute lookup, not lexical name lookup, but understanding the distinction is essential when choosing `self`, `cls`, or an unqualified name.


In [29]:
assert before == (10, 10, 10)
assert after == (50, 50, 10)


# Problem 17 — Multiple generated classes and closure isolation

Create three classes in a loop using a helper factory.  
Each class should retain its own label.

Avoid the classic late-binding closure mistake.


In [30]:
def make_labeled_class(label):
    class Labeled:
        @classmethod
        def label(cls):
            return label
    return Labeled

classes = [
    make_labeled_class(label)
    for label in ("A", "B", "C")
]

outputs = [cls.label() for cls in classes]
print(outputs)

assert outputs == ["A", "B", "C"]


['A', 'B', 'C']


### Solution 17

Calling a helper factory once per iteration creates a fresh function activation and therefore a fresh closure cell for `label`.

This is clearer than relying on tricky loop-capture workarounds inside a single enclosing scope.


# Problem 18 — Compare with late binding in ordinary closures

This is not class-specific, but it is valuable for distinguishing *class scope* issues from *closure* issues.

Why do all functions return the same value?


In [31]:
funcs = []

for i in range(3):
    def f():
        return i
    funcs.append(f)

print([f() for f in funcs])
assert [f() for f in funcs] == [2, 2, 2]


[2, 2, 2]


### Solution 18

Closures capture variables/cells, not frozen snapshots of their values.

A standard fix is to bind the current value as a default argument:


In [32]:
funcs_fixed = []

for i in range(3):
    def f(i=i):
        return i
    funcs_fixed.append(f)

print([f() for f in funcs_fixed])
assert [f() for f in funcs_fixed] == [0, 1, 2]


[0, 1, 2]


# Problem 19 — Inspect bytecode to distinguish unqualified names from attributes

Compare bytecode for:

- `return VALUE`
- `return self.VALUE`

Look for different load operations.


In [33]:
VALUE = "global"

class BytecodeDemo:
    VALUE = "class"

    def unqualified(self):
        return VALUE

    def qualified(self):
        return self.VALUE

print("unqualified():")
dis.dis(BytecodeDemo.unqualified)

print("\nqualified():")
dis.dis(BytecodeDemo.qualified)


unqualified():
  6           RESUME                   0

  7           LOAD_GLOBAL              0 (VALUE)
              RETURN_VALUE

qualified():
  9           RESUME                   0

 10           LOAD_FAST                0 (self)
              LOAD_ATTR                0 (VALUE)
              RETURN_VALUE


### Solution 19

Exact bytecode mnemonics can vary across Python versions, but conceptually:

- the unqualified version performs **name/global resolution**;
- the qualified version first loads `self`, then performs **attribute lookup**.

This is the operational difference behind many class-scope surprises.


# Problem 20 — Observe the class namespace while the body is running

Use `locals()` directly inside a class body to inspect names created so far.


In [34]:
class NamespaceProbe:
    first = 1
    snapshot_after_first = tuple(sorted(locals().keys()))

    second = 2
    snapshot_after_second = tuple(sorted(locals().keys()))

print("after first:")
pprint(NamespaceProbe.snapshot_after_first)

print("\nafter second:")
pprint(NamespaceProbe.snapshot_after_second)

assert "first" in NamespaceProbe.snapshot_after_first
assert "second" not in NamespaceProbe.snapshot_after_first
assert "second" in NamespaceProbe.snapshot_after_second


after first:
('__firstlineno__', '__module__', '__qualname__', 'first')

after second:
('__firstlineno__',
 '__module__',
 '__qualname__',
 'first',
 'second',
 'snapshot_after_first')


### Solution 20

The class body is executable code operating against a namespace mapping.  
Names appear as assignments execute.

Do not use mutation of `locals()` as ordinary application logic. Treat this as an introspection demonstration.


# Problem 21 — Static method, inheritance, and a misleading global

A static method cannot receive `self` or `cls` automatically.

Refactor the buggy implementation so subclasses get their own `PREFIX`.


In [35]:
PREFIX = "GLOBAL"

class Message:
    PREFIX = "BASE"

    @staticmethod
    def buggy(text):
        return f"{PREFIX}: {text}"

class WarningMessage(Message):
    PREFIX = "WARNING"

print(WarningMessage.buggy("disk nearly full"))


GLOBAL: disk nearly full


### Solution 21

If behavior should be polymorphic across subclasses, this should be a class method, not a static method.


In [36]:
class Message:
    PREFIX = "BASE"

    @classmethod
    def format(cls, text):
        return f"{cls.PREFIX}: {text}"

class WarningMessage(Message):
    PREFIX = "WARNING"

assert WarningMessage.format("disk nearly full") == "WARNING: disk nearly full"
print(WarningMessage.format("disk nearly full"))


# Problem 22 — Property lookup follows the same method-body rules

Fix the buggy property so it uses the class/instance attribute rather than the module global.


In [37]:
UNIT = "GLOBAL_UNIT"

class Measurement:
    UNIT = "kg"

    def __init__(self, value):
        self.value = value

    @property
    def label_buggy(self):
        return f"{self.value} {UNIT}"

m = Measurement(12)
print(m.label_buggy)


12 GLOBAL_UNIT


### Solution 22


In [38]:
class Measurement:
    UNIT = "kg"

    def __init__(self, value):
        self.value = value

    @property
    def label(self):
        return f"{self.value} {self.UNIT}"

assert Measurement(12).label == "12 kg"
print(Measurement(12).label)


12 kg


# Problem 23 — `cls` and subclass-specific mutation

Implement a counter where calling `Child.bump()` creates/updates `Child.count` without modifying `BaseCounter.count`.


In [39]:
class BaseCounter:
    count = 0

    @classmethod
    def bump(cls):
        cls.count += 1
        return cls.count

class ChildCounter(BaseCounter):
    pass

print("Base before:", BaseCounter.count)
print("Child before:", ChildCounter.count)

ChildCounter.bump()
ChildCounter.bump()

print("Base after:", BaseCounter.count)
print("Child after:", ChildCounter.count)

assert BaseCounter.count == 0
assert ChildCounter.count == 2


Base before: 0
Child before: 0
Base after: 0
Child after: 2


### Solution 23

`cls.count += 1` first reads through the class hierarchy and then assigns on `cls`.

So the first call sees inherited `0`, computes `1`, and stores `ChildCounter.count = 1`.  
The base class remains unchanged.


# Problem 24 — Build a tiny test helper for expected exceptions

Best practice in teaching notebooks: expected errors should not stop execution.

Create `expect_exception(exc_type, fn)` and use it to prove an unqualified class constant fails when no matching global exists.


In [40]:
def expect_exception(exc_type, fn):
    try:
        fn()
    except exc_type as exc:
        print(f"Caught expected {exc_type.__name__}: {exc}")
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}"
        ) from exc
    else:
        raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised")

globals().pop("MISSING_CONST", None)

class ErrorDemo:
    MISSING_CONST = 123

    @classmethod
    def bad(cls):
        return MISSING_CONST

expect_exception(NameError, ErrorDemo.bad)


Caught expected NameError: name 'MISSING_CONST' is not defined


NameError("name 'MISSING_CONST' is not defined")

### Solution 24

The helper makes negative tests explicit and lets the rest of the notebook continue.

In real automated tests, use the exception assertion facilities of `pytest` or `unittest`.


# Problem 25 — Comprehensive scope puzzle

Predict every entry in the output dictionary before running the cell.

This combines:

- module globals
- class-body execution
- method lookup
- classmethod lookup
- closure capture from a factory
- default-argument evaluation
- comprehension behavior


In [41]:
X = "G"

def build():
    X = "F"

    class Puzzle:
        X = "C"

        direct = X
        repeated = [X] * 2
        comprehension = [X for _ in range(2)]

        def method(self):
            return X

        @classmethod
        def class_attr(cls):
            return cls.X

        def defaulted(self, value=X):
            return value

    return Puzzle

Puzzle = build()

answer = {
    "class_attr_value": Puzzle.X,
    "direct": Puzzle.direct,
    "repeated": Puzzle.repeated,
    "comprehension": Puzzle.comprehension,
    "method": Puzzle().method(),
    "class_method": Puzzle.class_attr(),
    "defaulted": Puzzle().defaulted(),
}

pprint(answer)


{'class_attr_value': 'C',
 'class_method': 'C',
 'comprehension': ['F', 'F'],
 'defaulted': 'C',
 'direct': 'C',
 'method': 'F',
 'repeated': ['C', 'C']}


### Solution 25

Expected:

```python
{
    "class_attr_value": "C",
    "direct": "C",
    "repeated": ["C", "C"],
    "comprehension": ["F", "F"],
    "method": "F",
    "class_method": "C",
    "defaulted": "C",
}
```

Reasoning:

- `Puzzle.X`, `direct`, and `repeated` are class-body results.
- the comprehension body can close over the enclosing function's `X == "F"`;
- `method()` also closes over factory-local `X`;
- `class_attr()` explicitly uses `cls.X`;
- the default argument `value=X` is evaluated while the `def` statement executes in the class body, so it stores `"C"`.


In [42]:
expected = {
    "class_attr_value": "C",
    "direct": "C",
    "repeated": ["C", "C"],
    "comprehension": ["F", "F"],
    "method": "F",
    "class_method": "C",
    "defaulted": "C",
}

assert answer == expected
print("Comprehensive puzzle passed.")


Comprehensive puzzle passed.


# Problem 26 — Refactor for clarity

The previous puzzle is valid Python, but much of its behavior is too subtle for ordinary application code.

Refactor the design so all runtime configuration is obtained explicitly through class attributes.


In [43]:
def build_clear(x):
    class ClearPuzzle:
        X = x

        @classmethod
        def direct_value(cls):
            return cls.X

        @classmethod
        def repeated(cls, n=2):
            return [cls.X] * n

        def method(self):
            return self.X

        @classmethod
        def class_attr(cls):
            return cls.X

        def defaulted(self, value=None):
            if value is None:
                value = self.X
            return value

    return ClearPuzzle

ClearPuzzle = build_clear("C")

assert ClearPuzzle.direct_value() == "C"
assert ClearPuzzle.repeated() == ["C", "C"]
assert ClearPuzzle().method() == "C"
assert ClearPuzzle.class_attr() == "C"
assert ClearPuzzle().defaulted() == "C"

print("Clear refactor passed.")


Clear refactor passed.


### Solution 26

The refactor makes the source of configuration visible:

- runtime instance behavior → `self.X`
- class-level polymorphic behavior → `cls.X`
- class construction parameter → copied intentionally into `X`

This is easier to maintain and safer under inheritance.


# Rapid-fire mini drills

Try to answer each mentally before running it.


In [44]:
# Drill A: class-body dependency order
class A:
    first = 10
    second = first + 5

assert A.second == 15

# Drill B: earlier names are available; later names are not.
try:
    class B:
        early = later
        later = 20
except NameError as exc:
    print("Drill B:", exc)

# Drill C: instance lookup can inherit from the class.
class C:
    value = 7

assert C().value == 7

# Drill D: instance assignment can shadow the class attribute.
c = C()
c.value = 9

assert c.value == 9
assert C.value == 7

# Drill E: subclass classmethod dispatch.
class ParentKind:
    kind = "parent"

    @classmethod
    def get_kind(cls):
        return cls.kind

class ChildKind(ParentKind):
    kind = "child"

assert ChildKind.get_kind() == "child"

print("Rapid-fire drills passed.")


Drill B: name 'later' is not defined
Rapid-fire drills passed.


# Debugging lab — Find and fix all scope smells

The following design has several issues:

1. a method reads a same-named global instead of the class attribute;
2. a static method should probably be polymorphic;
3. a default argument freezes configuration too early;
4. a comprehension relies on a global with the same name as a class constant.

First observe the buggy behavior, then compare with the corrected design.


In [45]:
MODE = "global-mode"
PREFIX = "GLOBAL"
MULTIPLIER = 100

class BuggyService:
    MODE = "class-mode"
    PREFIX = "SERVICE"
    MULTIPLIER = 3

    scaled = [MULTIPLIER * i for i in range(3)]

    def mode(self):
        return MODE

    @staticmethod
    def format_message(text):
        return f"{PREFIX}: {text}"

    def compute(self, multiplier=MULTIPLIER):
        return 10 * multiplier

print("mode:", BuggyService().mode())
print("message:", BuggyService.format_message("hello"))
print("compute:", BuggyService().compute())
print("scaled:", BuggyService.scaled)


mode: global-mode
message: GLOBAL: hello
compute: 30
scaled: [0, 100, 200]


## Debugging lab solution


In [46]:
class Service:
    MODE = "class-mode"
    PREFIX = "SERVICE"
    MULTIPLIER = 3

    def mode(self):
        return self.MODE

    @classmethod
    def format_message(cls, text):
        return f"{cls.PREFIX}: {text}"

    def compute(self, multiplier=None):
        if multiplier is None:
            multiplier = self.MULTIPLIER
        return 10 * multiplier

    @classmethod
    def build_scaled(cls):
        return [cls.MULTIPLIER * i for i in range(3)]

assert Service().mode() == "class-mode"
assert Service.format_message("hello") == "SERVICE: hello"
assert Service().compute() == 30
assert Service.build_scaled() == [0, 3, 6]

class FastService(Service):
    MODE = "fast"
    PREFIX = "FAST"
    MULTIPLIER = 8

assert FastService().mode() == "fast"
assert FastService.format_message("hello") == "FAST: hello"
assert FastService().compute() == 80
assert FastService.build_scaled() == [0, 8, 16]

print("Corrected design passes all checks.")


Corrected design passes all checks.


# Capstone challenge — Scope-safe plugin registry

Build a base class whose subclasses can override configuration without being broken by globals.

Requirements:

- `Plugin.NAME` is a class attribute.
- `qualified_name()` must be polymorphic.
- `run()` should use instance/class configuration explicitly.
- `make_plugin(name, weight)` should return a generated subclass.
- The generated subclass may store factory arguments as class attributes, but runtime methods should read through `self` or `cls`.
- Include assertions for at least three plugins.


In [47]:
class Plugin:
    NAME = "base"
    WEIGHT = 1

    @classmethod
    def qualified_name(cls):
        return f"plugin:{cls.NAME}"

    def run(self, value):
        return {
            "plugin": self.NAME,
            "input": value,
            "weighted": value * self.WEIGHT,
        }


def make_plugin(name, weight):
    class GeneratedPlugin(Plugin):
        NAME = name
        WEIGHT = weight

    # Optional cosmetic improvement for debugging.
    GeneratedPlugin.__name__ = f"{name.title()}Plugin"
    return GeneratedPlugin


AlphaPlugin = make_plugin("alpha", 2)
BetaPlugin = make_plugin("beta", 5)
GammaPlugin = make_plugin("gamma", -1)

assert AlphaPlugin.qualified_name() == "plugin:alpha"
assert BetaPlugin.qualified_name() == "plugin:beta"
assert GammaPlugin.qualified_name() == "plugin:gamma"

assert AlphaPlugin().run(10)["weighted"] == 20
assert BetaPlugin().run(10)["weighted"] == 50
assert GammaPlugin().run(10)["weighted"] == -10

for plugin_cls in (AlphaPlugin, BetaPlugin, GammaPlugin):
    print(plugin_cls.__name__, plugin_cls.qualified_name(), plugin_cls().run(4))


AlphaPlugin plugin:alpha {'plugin': 'alpha', 'input': 4, 'weighted': 8}
BetaPlugin plugin:beta {'plugin': 'beta', 'input': 4, 'weighted': 20}
GammaPlugin plugin:gamma {'plugin': 'gamma', 'input': 4, 'weighted': -4}


## Capstone solution discussion

The factory uses lexical scope only during class construction to copy `name` and `weight` into class attributes.

After construction, normal runtime behavior goes through `cls.NAME`, `self.NAME`, and `self.WEIGHT`. This gives subclasses and later overrides predictable semantics.

That separation is a useful design principle:

> **Capture configuration deliberately; perform runtime behavior through explicit attributes.**


# Final review: what resolves where?

| Code location | Recommended way to access class/instance data | Why |
|---|---|---|
| Direct class body | earlier class-body name may be referenced directly | class namespace is currently executing |
| Instance method | `self.attr` | supports instance shadowing + inheritance |
| Class method | `cls.attr` | supports subclass polymorphism |
| Static method | explicit dependency / explicit class only if truly fixed | no automatic `self` or `cls` |
| Function inside class | do not expect class locals as lexical parents | function scope skips class namespace |
| Method in class factory | closure variables are possible | enclosing **function** is lexical scope |
| Comprehension in class body | avoid depending on class locals in comprehension body | comprehension has its own function-like scope |
| Method needing runtime default | sentinel + `self.attr`/`cls.attr` | avoids frozen default values |
| Defining class reference | `__class__` when specifically appropriate | special compiler-supported cell |


# Final self-test

You should now be able to explain all of the following without running code:

1. Why a class body can use an earlier class variable directly.
2. Why a method usually cannot use that same class variable as an unqualified name.
3. Why a matching global can turn a `NameError` into a silent logic bug.
4. Why `cls.attr` is usually preferable in a class method.
5. Why a method in a factory-generated class can be a closure.
6. Why a class-body list comprehension can behave differently from a direct list expression.
7. Why a default argument can capture a class-body value at definition time.
8. Why nested classes do not automatically capture outer-class attributes.
9. Why `__class__` is special.
10. Why explicit attribute access generally produces more maintainable code.


In [48]:
# Final smoke test: if this cell runs, the notebook reached the end successfully.
print("All notebook sections loaded successfully.")


All notebook sections loaded successfully.
